In [1]:
import torch

In [2]:
import gpuclustersim

In [3]:
from torch.utils.flop_counter import FlopCounterMode

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("gpuclustersim")

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 1)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 input channel, 16 output channels, 3x3 square convolution
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 7 * 7, 10) # Assumes a 28x28 input image

    def forward(self, x):
        # x expected shape: (batch_size, 1, 28, 28)
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        return self.fc(x)

class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=1, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        # x expected shape: (batch_size, sequence_length, 10)
        out, hidden = self.rnn(x)
        # Slicing: take the output of the last time step
        last_out = out[:, -1, :] 
        return self.fc(last_out)

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=1, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        # x expected shape: (batch_size, sequence_length, 10)
        out, (hn, cn) = self.lstm(x) # LSTM returns output and (hidden_state, cell_state)
        # Slicing: take the output of the last time step
        last_out = out[:, -1, :]
        return self.fc(last_out)

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # A single transformer encoder block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=16, 
            nhead=2, 
            dim_feedforward=32, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(16, 2)

    def forward(self, x):
        # x expected shape: (batch_size, sequence_length, 16)
        out = self.transformer(x)
        # Aggregate over the sequence length (Global Average Pooling)
        out = out.mean(dim=1) 
        return self.fc(out)

class SimpleResidual(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 10)
        self.fc2 = nn.Linear(10, 10)

    def forward(self, x):
        # x expected shape: (batch_size, 10)
        identity = x
        out = torch.relu(self.fc1(x))
        out = self.fc2(out)
        out += identity # Explicit element-wise addition
        return torch.relu(out)

In [5]:
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [6]:
with FlopCounterMode(display=True) as flop_counter:
  for step in range(1):
    x = torch.randn(32, 10).to(device)
    y = torch.randn(32, 1).to(device)
    optimizer.zero_grad()
    print("FORWARDPROP")
    loss = criterion(model(x), y)
    print("BACKWARDPROP")
    loss.backward()
    print("OPTSTEP")
    optimizer.step()

FORWARDPROP
Op Name: aten::addmm
Op Overload Name: out
Op Inputs: Tensor self Tensor mat1 Tensor mat2 Scalar beta=1 Scalar alpha=1 Tensor(a!) out
Op Outputs: Tensor(a!)
Op Name: aten::relu
Op Overload Name: 
Op Inputs: Tensor self
Op Outputs: Tensor
Op Name: aten::addmm
Op Overload Name: out
Op Inputs: Tensor self Tensor mat1 Tensor mat2 Scalar beta=1 Scalar alpha=1 Tensor(a!) out
Op Outputs: Tensor(a!)
Op Name: aten::relu
Op Overload Name: 
Op Inputs: Tensor self
Op Outputs: Tensor
Op Name: aten::addmm
Op Overload Name: out
Op Inputs: Tensor self Tensor mat1 Tensor mat2 Scalar beta=1 Scalar alpha=1 Tensor(a!) out
Op Outputs: Tensor(a!)
Op Name: aten::mse_loss
Op Overload Name: out
Op Inputs: Tensor self Tensor target int reduction=1 Tensor(a!) out
Op Outputs: Tensor(BACKWARDPROP
a!)
Op Name: aten::fill_
Op Overload Name: Scalar
Op Inputs: Tensor(a!) self Scalar value
Op Outputs: Tensor(a!)
Op Name: aten::mse_loss_backward
Op Overload Name: 
Op Inputs: Tensor grad_output Tensor self Te

In [6]:
model = SimpleCNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [7]:
for steps in range(1):
  x = torch.randn(32, 1, 28, 28, device=device)
  y = torch.randn(32, 10, device=device)

  optimizer.zero_grad()
  print("FORWARDPROP")
  loss = criterion(model(x), y)
  print("BACKWARDPROP")
  loss.backward()
  print("OPTSTEP")
  optimizer.step()

Op Name: aten::normal_
Op Name: aten::normal_
FORWARDPROP
Op Name: aten::convolution_overrideable
Op Name: aten::relu
Op Name: aten::max_pool2d_with_indices
Op Name: aten::convolution_overrideable
Op Name: aten::relu
Op Name: aten::max_pool2d_with_indices
Op Name: aten::addmm
Op Name: aten::mse_loss
BACKWARDPROP
Op Name: aten::fill_
Op Name: aten::mse_loss_backward
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::max_pool2d_with_indices_backward
Op Name: aten::threshold_backward
Op Name: aten::convolution_backward_overrideable
Op Name: aten::max_pool2d_with_indices_backward
Op Name: aten::threshold_backward
Op Name: aten::convolution_backward_overrideable
OPTSTEP
Op Name: aten::add
Op Name: aten::add
Op Name: aten::add
Op Name: aten::add
Op Name: aten::add
Op Name: aten::add


In [8]:
# --- SimpleRNN ---
model = SimpleRNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


In [9]:
for step in range(1):
  x = torch.randn(16, 8, 10).to(device)   # (batch, seq_len, input_size=10)
  y = torch.randn(16, 2).to(device)        # (batch, out_features=2)
  optimizer.zero_grad()
  print("FORWARDPROP")
  loss = criterion(model(x), y)
  print("BACKWARDPROP")
  loss.backward()
  print("OPTSTEP")
  optimizer.step()


FORWARDPROP
Op Name: aten::zero_
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::addmm
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::tanh
Op Name: aten::cat
Op Name: aten::cat
Op Name: aten::addmm
Op Name: aten::mse_loss
BACKWARDPROP
Op Name: aten::fill_
Op Name: aten::mse_loss_backward
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::zero_
Op Name: aten::tanh_backward
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::mm
Op Name: 

In [10]:
# --- SimpleLSTM ---
model = SimpleLSTM().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


In [11]:
for step in range(1):
  x = torch.randn(16, 8, 10).to(device)   # (batch, seq_len, input_size=10)
  y = torch.randn(16, 2).to(device)
  optimizer.zero_grad()
  print("FORWARDPROP")
  loss = criterion(model(x), y)
  print("BACKWARDPROP")
  loss.backward()
  print("OPTSTEP")
  optimizer.step()


FORWARDPROP
Op Name: aten::zero_
Op Name: aten::zero_
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::_thnn_fused_lstm_cell
Op Name: aten::cat
Op Name: aten::cat
Op Name: aten::cat
Op Name: aten::addmm
Op Name: aten::mse_loss
BACKWARDPROP
Op Name: aten::fill_
Op Name: aten::mse_loss_backward
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::zero_
Op Name: aten::_thnn_fused_lstm_cell_backward_impl
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::add
Op Name: at

In [12]:
# --- SimpleTransformer ---
model = SimpleTransformer().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


In [13]:
for step in range(1):
  x = torch.randn(16, 8, 16).to(device)   # (batch, seq_len, d_model=16)
  y = torch.randn(16, 2).to(device)
  optimizer.zero_grad()
  print("FORWARDPROP")
  loss = criterion(model(x), y)
  print("BACKWARDPROP")
  loss.backward()
  print("OPTSTEP")
  optimizer.step()


FORWARDPROP
Op Name: aten::mm
Op Name: aten::add
Op Name: aten::mul
Op Name: aten::mul
Op Name: aten::bmm
Op Name: aten::_softmax
Op Name: aten::isneginf
Op Name: aten::all
Op Name: aten::fill_
Op Name: aten::where
Op Name: aten::native_dropout
Op Name: aten::bmm
Op Name: aten::addmm
Op Name: aten::native_dropout
Op Name: aten::add
Op Name: aten::native_batch_norm
Op Name: aten::addcmul
Op Name: aten::addmm
Op Name: aten::relu
Op Name: aten::native_dropout
Op Name: aten::addmm
Op Name: aten::native_dropout
Op Name: aten::add
Op Name: aten::native_batch_norm
Op Name: aten::addcmul
Op Name: aten::mean
Op Name: aten::addmm
Op Name: aten::mse_loss
BACKWARDPROP
Op Name: aten::fill_
Op Name: aten::mse_loss_backward
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::div
Op Name: aten::native_layer_norm_backward
Op Name: aten::native_dropout_backward
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::native_dropout_backward
Op Name: aten::threshold_backwar

In [14]:
# --- SimpleResidual ---
model = SimpleResidual().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


In [15]:
for step in range(1):
  x = torch.randn(16, 10).to(device)       # (batch, 10)
  y = torch.randn(16, 10).to(device)
  optimizer.zero_grad()
  print("FORWARDPROP")
  loss = criterion(model(x), y)
  print("BACKWARDPROP")
  loss.backward()
  print("OPTSTEP")
  optimizer.step()


FORWARDPROP
Op Name: aten::addmm
Op Name: aten::relu
Op Name: aten::addmm
Op Name: aten::add
Op Name: aten::relu
Op Name: aten::mse_loss
BACKWARDPROP
Op Name: aten::fill_
Op Name: aten::mse_loss_backward
Op Name: aten::threshold_backward
Op Name: aten::mm
Op Name: aten::mm
Op Name: aten::sum
Op Name: aten::threshold_backward
Op Name: aten::mm
Op Name: aten::sum
OPTSTEP
Op Name: aten::add
Op Name: aten::add
Op Name: aten::add
Op Name: aten::add
